# FloraSurvival AI — Pipeline ML Lengkap (Tahap 0–6)

Klasifikasi 100 spesies tanaman obat Indonesia dari citra, sampai model
`.tflite` siap dipakai aplikasi. Notebook ini **satu-satunya file notebook**
dan sudah dieksekusi penuh dari atas ke bawah; semua output sel di bawah
berasal dari run nyata, bukan contoh.

| | |
|---|---|
| Dataset | 10.000 gambar, 100 spesies, asli 128×128 |
| Model final | MobileNetV2 fine-tune, input 224×224, 2,39 juta parameter |
| Hasil test | top-1 **89,16%**, top-3 **95,26%** |
| Dengan `AMBANG = 0.27` | 93% foto dijawab, akurasi **92,7%** |
| Artefak | `model.tflite` **4,56 MB**, float16 |
| Waktu eksekusi | ± 80 menit, CPU 4 core, tanpa GPU |

> **Soal variasi angka.** Augmentasi gambar memakai RNG sehingga run ulang
> dengan kode dan seed yang sama masih bisa bergeser (run sebelumnya mendapat
> top-1 85,03%). Semua angka di notebook ini berasal dari eksekusi ini, dan
> angka di laporan sudah disamakan dengan hasil ini.

## Isi

| Tahap | Kode | Waktu |
|---|---|---|
| 0–1 | `00_prepare.py` — bersihkan data + split | 2 mnt |
| 2 | `01_train_cnn.py` — model final 224px | 60 mnt |
| 3 | `02_svm.py` — model 2 | 1 mnt |
| 4 | `03_rf.py` — model 3 | 1 mnt |
| 5 | `04_compare.py` — perbandingan | 10 dtk |
| 6 | `05_export_tflite.py` — export + verifikasi | 12 mnt |
| 6a | `06_threshold.py` — ambang penolakan | 5 dtk |
| 6b | `06b_svm_deploy.py` — biaya deploy SVM | 2 mnt |

## Catatan

- Semua path absolut ke folder `tanaman-obat`. Kalau folder dipindah, ubah di
  sel pertama tiap tahap.
- Menjalankan ulang notebook ini **menimpa** folder `output/` (kecuali file
  laporan yang ditulis manual). Angka bisa bergeser sedikit karena augmentasi
  acak.
- Tanpa GPU. Pastikan laptop tidak sleep saat training.
- Grafik disimpan sebagai PNG di `figures/` tiap tahap, bukan ditampilkan
  inline, karena matplotlib memakai backend `Agg`.
- Percobaan 128px dan TTA tidak ada di sini: keduanya tidak dipakai di model
  final, hanya jadi pembanding di laporan.


---

# Tahap 0–1 · Persiapan Data

Bersihkan dataset lalu bagi jadi train/val/test.

**Masuk:** 10.000 gambar, 100 folder spesies, semua 128×128.
**Keluar:** `output/_data/` (`labels.json`, `train.csv`, `val.csv`, `test.csv`,
`dataset_report.csv`).

Dua lapis pembersihan: duplikat persis (MD5) dan near-duplicate dalam kelas
(dHash ≤ 2). Duplikat dibuang **sebelum** split supaya tidak ada gambar yang
sama muncul di train dan di test.

Hasil: 10.000 → 9.839 gambar (6.887 / 1.476 / 1.476).

In [1]:
import hashlib
import json
import sys
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageFile
from sklearn.model_selection import train_test_split

ImageFile.LOAD_TRUNCATED_IMAGES = True

sys.path.insert(0, r"C:\Users\Wahyu Pratomo\Project Machine Learning\tanaman-obat\scripts")
from common import DATA, OUT  # noqa: E402

ROOT = Path(r"C:\Users\Wahyu Pratomo\Downloads\Indonesia-Medicinal-Plant-dataset\Indonesia-Medicinal-Plant-Dataset")
NEAR_DUP_MAX = 2

In [2]:
def md5(p):
    return hashlib.md5(Path(p).read_bytes()).hexdigest()


def dhash(p, size=8):
    with Image.open(p) as im:
        a = np.asarray(im.convert("L").resize((size + 1, size), Image.BILINEAR), dtype=np.int16)
    return (a[:, 1:] > a[:, :-1]).flatten()

In [3]:
print("=== TAHAP 0-1: cek dataset + pembersihan ===", flush=True)
rows = []
for d in sorted(p for p in ROOT.iterdir() if p.is_dir()):
    for f in sorted(d.iterdir()):
        if f.is_file() and f.suffix.lower() == ".jpg":
            rows.append({"path": str(f), "label": d.name})
df = pd.DataFrame(rows)
print(f"file terbaca: {len(df):,} | folder: {df.label.nunique()}", flush=True)

=== TAHAP 0-1: cek dataset + pembersihan ===


file terbaca: 10,000 | folder: 100


In [4]:
labels = sorted(df.label.unique())
idx = {c: i for i, c in enumerate(labels)}
df["y"] = df.label.map(idx)
(DATA / "labels.json").write_text(json.dumps(labels, indent=1, ensure_ascii=False), encoding="utf-8")

3434

In [5]:
# --- duplikat persis ---
print("hitung MD5...", flush=True)
df["md5"] = [md5(p) for p in df.path]
n0 = len(df)
df = df.drop_duplicates("md5", keep="first").reset_index(drop=True)
exact_removed = n0 - len(df)
print(f"duplikat persis dibuang: {exact_removed}", flush=True)

hitung MD5...


duplikat persis dibuang: 69


In [6]:
# --- near duplicate dalam kelas ---
print("hitung dHash...", flush=True)
df["hash"] = [dhash(p) for p in df.path]
keep = np.ones(len(df), dtype=bool)
by_class = defaultdict(list)
for i, lab in enumerate(df.label.values):
    by_class[lab].append(i)
near_removed = 0
for lab, ids in by_class.items():
    kept_idx = []
    for i in ids:
        dup = False
        for j in kept_idx:
            if int((df.hash.values[i] != df.hash.values[j]).sum()) <= NEAR_DUP_MAX:
                dup = True
                break
        if dup:
            keep[i] = False
            near_removed += 1
        else:
            kept_idx.append(i)
df = df[keep].drop(columns=["hash"]).reset_index(drop=True)
print(f"near-duplicate dibuang: {near_removed}", flush=True)

hitung dHash...


near-duplicate dibuang: 92


In [7]:
# --- split stratified ---
tr, rest = train_test_split(df, test_size=0.30, random_state=42, stratify=df.y)
va, te = train_test_split(rest, test_size=0.50, random_state=42, stratify=rest.y)
for name, part in (("train", tr), ("val", va), ("test", te)):
    part[["path", "label", "y"]].to_csv(DATA / f"{name}.csv", index=False)
    print(f"{name:5} {len(part):5,} gambar", flush=True)

train 6,887 gambar


val   1,476 gambar


test  1,476 gambar


In [8]:
report = pd.DataFrame({
    "item": ["file awal", "duplikat persis dibuang", "near-duplicate dibuang",
             "file bersih", "train", "val", "test", "kelas"],
    "nilai": [n0, exact_removed, near_removed, len(df), len(tr), len(va), len(te), len(labels)],
})
report.to_csv(DATA / "dataset_report.csv", index=False)
print(report.to_string(index=False))
print("DONE TAHAP 0-1", flush=True)

                   item  nilai
              file awal  10000
duplikat persis dibuang     69
 near-duplicate dibuang     92
            file bersih   9839
                  train   6887
                    val   1476
                   test   1476
                  kelas    100
DONE TAHAP 0-1


---

# Tahap 2 · Model 1 — CNN MobileNetV2 224px (model final)

Fine-tune MobileNetV2 ImageNet untuk 100 kelas, dua fase: kepala
dilatih dulu (base beku), lalu 40 lapisan terakhir di-unfreeze. BatchNorm
tetap beku di fase fine-tune.

Parameter run final: `img=224, head=3, fine=15, batch=16, dropout=0.4,
label_smoothing=0.1`. Regularisasi sekuat ini dipakai karena run 128px
overfit (train 94% vs val 79%).

Normalisasi `Rescaling(1/127.5, -1)` dipasang **di dalam** model — itu sebabnya
aplikasi mengirim piksel 0–255 apa adanya.

**Keluar:** `output/01_cnn/` → `model.keras`, `metrics.json`, `HASIL.md`,
4 grafik, plus embedding dan probabilitas untuk model 2 dan 3.

In [9]:
import argparse
import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf

sys.path.insert(0, r"C:\Users\Wahyu Pratomo\Project Machine Learning\tanaman-obat\scripts")
from common import (DATA, OUT, abstain_curve, load_tags, safety_metrics,  # noqa: E402
                    save_json, topk_acc, worst_classes, write_md)

import matplotlib  # noqa: E402
matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402

tf.random.set_seed(42)
np.random.seed(42)

args = argparse.Namespace(img=224, head=3, fine=15, batch=16, out='01_cnn', dropout=0.4, ls=0.1, lr_fine=0.0003, reuse=False)

D = OUT / args.out
FIG = D / "figures"
FIG.mkdir(parents=True, exist_ok=True)
IMG = args.img
BATCH = args.batch

In [10]:
labels = json.loads((DATA / "labels.json").read_text(encoding="utf-8"))
N = len(labels)
tr = pd.read_csv(DATA / "train.csv")
va = pd.read_csv(DATA / "val.csv")
te = pd.read_csv(DATA / "test.csv")
print(f"train {len(tr):,} | val {len(va):,} | test {len(te):,} | kelas {N} | img {IMG}", flush=True)

train 6,887 | val 1,476 | test 1,476 | kelas 100 | img 224


In [11]:
def decode(path, y):
    raw = tf.io.read_file(path)
    img = tf.io.decode_image(raw, channels=3, expand_animations=False)
    img = tf.image.resize(img, [IMG, IMG])
    img = tf.ensure_shape(img, [IMG, IMG, 3])
    return tf.cast(img, tf.float32), y


aug = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.12),
    tf.keras.layers.RandomZoom(0.12, 0.10),
    tf.keras.layers.RandomContrast(0.15),
    tf.keras.layers.RandomBrightness(0.12),
], name="aug")

In [12]:
def make_ds(frame, training, shuffle=2048):
    ds = tf.data.Dataset.from_tensor_slices((frame.path.values, frame.y.values))
    if training:
        ds = ds.shuffle(shuffle, seed=42, reshuffle_each_iteration=True)
    ds = ds.map(decode, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.map(lambda x, y: (aug(x, training=True), y),
                    num_parallel_calls=tf.data.AUTOTUNE)
    if args.ls > 0:
        ds = ds.map(lambda x, y: (x, tf.one_hot(y, N)),
                    num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH).prefetch(tf.data.AUTOTUNE)

In [13]:
def build():
    inp = tf.keras.Input((IMG, IMG, 3))
    x = tf.keras.layers.Rescaling(1.0 / 127.5, offset=-1.0, name="preprocess")(inp)
    base = tf.keras.applications.MobileNetV2(
        include_top=False, weights="imagenet", input_shape=(IMG, IMG, 3))
    base.trainable = False
    x = base(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D(name="embedding")(x)
    feats = x
    x = tf.keras.layers.Dropout(args.dropout)(x)
    out = tf.keras.layers.Dense(N, activation="softmax", name="logits")(x)
    return tf.keras.Model(inp, out), tf.keras.Model(inp, feats), base

In [14]:
ds_tr = make_ds(tr, True)
ds_va = make_ds(va, False)
ds_te = make_ds(te, False)

In [15]:
model_path = D / "model.keras"
if args.reuse and model_path.exists():
    print("reuse model lama", flush=True)
    model = tf.keras.models.load_model(model_path)
    base = [l for l in model.layers if l.name.startswith("mobilenet")][0]
    feat_model = tf.keras.Model(model.input, model.get_layer("embedding").output)
    hist = pd.read_csv(D / "history.csv")
else:
    model, feat_model, base = build()
    LOSS = (tf.keras.losses.CategoricalCrossentropy(label_smoothing=args.ls)
            if args.ls > 0 else "sparse_categorical_crossentropy")
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                  loss=LOSS, metrics=["accuracy"])
    print(f"parameter total: {model.count_params():,}", flush=True)

    t0 = time.time()
    h1 = model.fit(ds_tr, validation_data=ds_va, epochs=args.head, verbose=2)
    print(f"fase head: {(time.time()-t0)/60:.1f} menit", flush=True)

    base.trainable = True
    for layer in base.layers[:-40]:
        layer.trainable = False
    for layer in base.layers:
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False
    model.compile(optimizer=tf.keras.optimizers.Adam(args.lr_fine),
                  loss=LOSS, metrics=["accuracy"])
    print(f"parameter trainable: {sum(np.prod(w.shape) for w in model.trainable_weights):,}", flush=True)

    t0 = time.time()
    h2 = model.fit(ds_tr, validation_data=ds_va, epochs=args.fine, verbose=2,
                   callbacks=[
                       tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=3,
                                                        restore_best_weights=True),
                       tf.keras.callbacks.ReduceLROnPlateau(monitor="val_accuracy", factor=0.5,
                                                           patience=2, min_lr=1e-6),
                   ])
    print(f"fase fine-tune: {(time.time()-t0)/60:.1f} menit", flush=True)
    hist = pd.concat([pd.DataFrame(h1.history), pd.DataFrame(h2.history)], ignore_index=True)
    hist.to_csv(D / "history.csv", index=False)
    model.save(model_path)

parameter total: 2,386,084


Epoch 1/3


  self.data_adapter = data_adapters.get_data_adapter(


431/431 - 197s - 457ms/step - accuracy: 0.4087 - loss: 2.9071 - val_accuracy: 0.5982 - val_loss: 2.2086


Epoch 2/3


431/431 - 176s - 409ms/step - accuracy: 0.6553 - loss: 1.9938 - val_accuracy: 0.6667 - val_loss: 2.0024


... (31 outputs left out)

431/431 - 212s - 492ms/step - accuracy: 0.9933 - loss: 0.8974 - val_accuracy: 0.8923 - val_loss: 1.2183 - learning_rate: 7.5000e-05


Epoch 15/15


431/431 - 211s - 489ms/step - accuracy: 0.9952 - loss: 0.8925 - val_accuracy: 0.8896 - val_loss: 1.2314 - learning_rate: 7.5000e-05


fase fine-tune: 53.1 menit


In [16]:
# ---------- evaluasi ----------
print("prediksi test...", flush=True)
probs = model.predict(ds_te, verbose=0)

prediksi test...


In [17]:
probs_va = model.predict(ds_va, verbose=0)
emb_tr = feat_model.predict(make_ds(tr, False), batch_size=64, verbose=0)
emb_va = feat_model.predict(ds_va, batch_size=64, verbose=0)
emb_te = feat_model.predict(ds_te, batch_size=64, verbose=0)

y_te, y_va, y_tr = te.y.values, va.y.values, tr.y.values
pred = probs.argmax(1)

In [18]:
np.save(D / "probabilitas_test.npy", probs)
np.save(D / "probabilitas_val.npy", probs_va)
np.save(D / "pred_test.npy", pred)

In [19]:
np.save(D / "emb_train.npy", emb_tr)
np.save(D / "emb_val.npy", emb_va)
np.save(D / "emb_test.npy", emb_te)

In [20]:
np.save(D / "y_train.npy", y_tr)
np.save(D / "y_val.npy", y_va)
np.save(D / "y_test.npy", y_te)

from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, recall_score  # noqa: E402

top1 = accuracy_score(y_te, pred)
top3 = topk_acc(y_te, probs, 3)
macro = f1_score(y_te, pred, average="macro")
rec_bucket = recall_score(y_te, pred, average=None, labels=list(range(N)))

In [21]:
tags = load_tags()
safe, fatal = safety_metrics(y_te, pred, labels, tags)

metrics = {"model": "CNN MobileNetV2", "img": IMG, "n_params": int(model.count_params()),
           "top1": float(top1), "top3": float(top3), "macro_f1": float(macro),
           **safe}

In [22]:
save_json(D / "metrics.json", metrics)

print(f"\nTOPI-1 {top1:.4f} | TOP-3 {top3:.4f} | macroF1 {macro:.4f}", flush=True)
print(f"fatal {safe['fatal_n']}/{safe['n_test_beracun']} = {safe['fatal_rate']:.1%}", flush=True)

  json -> C:\Users\Wahyu Pratomo\Project Machine Learning\tanaman-obat\output\01_cnn\metrics.json

TOPI-1 0.8916 | TOP-3 0.9526 | macroF1 0.8881


fatal 19/444 = 4.3%


In [23]:
# ---------- figur ----------
if "accuracy" in hist.columns:
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(hist["accuracy"], label="train", marker="o", ms=3)
    ax[0].plot(hist["val_accuracy"], label="val", marker="o", ms=3)
    ax[0].set_title("Akurasi per epoch"); ax[0].set_xlabel("epoch"); ax[0].grid(alpha=.3); ax[0].legend()
    ax[1].plot(hist["loss"], label="train", marker="o", ms=3)
    ax[1].plot(hist["val_loss"], label="val", marker="o", ms=3)
    ax[1].set_title("Loss per epoch"); ax[1].set_xlabel("epoch"); ax[1].grid(alpha=.3); ax[1].legend()
    plt.tight_layout(); plt.savefig(FIG / "kurva_training.png", dpi=130); plt.close()

In [24]:
cm = confusion_matrix(y_te, pred, labels=list(range(N)))
fig, ax = plt.subplots(figsize=(12, 11))
ax.imshow(cm, cmap="viridis")
ax.set_title(f"Confusion matrix CNN ({N} kelas)"); ax.set_xlabel("prediksi"); ax.set_ylabel("aktual")
plt.tight_layout(); plt.savefig(FIG / "confusion_matrix.png", dpi=110); plt.close()

In [25]:
per_class = cm.diagonal() / np.maximum(cm.sum(1), 1)
order = np.argsort(per_class)
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(len(per_class)), per_class[order], color="#2e7d32")
ax.set_title("Recall per kelas (urut naik)"); ax.set_xlabel("kelas"); ax.set_ylabel("recall")
ax.grid(axis="y", alpha=.3)
plt.tight_layout(); plt.savefig(FIG / "recall_per_kelas.png", dpi=120); plt.close()

In [26]:
cur = abstain_curve(y_te, pred, probs.max(1))
cur.to_csv(D / "kurva_abstain.csv", index=False)
fig, ax = plt.subplots(figsize=(7, 4.4))
ax.plot(cur.coverage * 100, cur.accuracy * 100, marker="o", color="#1565c0")
ax.set_xlabel("Foto yang dijawab (%)"); ax.set_ylabel("Akurasi pada yang dijawab (%)")
ax.set_title("Kalau model ragu, jangan dijawab"); ax.invert_xaxis(); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig(FIG / "kurva_abstain.png", dpi=130); plt.close()

In [27]:
wk, best = worst_classes(y_te, pred, labels, 15)
wk.to_csv(D / "kelas_terburuk.csv", index=False)

# ---------- HASIL.md ----------

In [28]:
tbl_abs = "\n".join(f"| {r.coverage:.0%} | {r.n} | {r.accuracy:.1%} |" for r in cur.itertuples())
tbl_worst = "\n".join(f"| {r.spesies} | {r.recall:.2f} | {r.n} |" for r in wk.itertuples())
tbl_best = "\n".join(f"| {r.spesies} | {r.recall:.2f} |" for r in best.itertuples())

In [29]:
sections = [
    ("Ringkasan", f"""| Metrik | Nilai |
|---|---|
| Top-1 | **{top1:.1%}** |
| Top-3 | **{top3:.1%}** |
| Macro-F1 | {macro:.1%} |
| Resolusi input | {IMG}x{IMG} |
| Parameter | {model.count_params():,} |
| Ukuran model | {model_path.stat().st_size/1024**2:.1f} MB |"""),
    ("Keamanan", f"""| Metrik | Nilai |
|---|---|
| Foto beracun di test | {safe['n_test_beracun']} |
| Dikenali sebagai beracun | {safe['recall_beracun_sebagai_beracun']:.1%} |
| **Kesalahan fatal (beracun -> spesies pangan)** | **{safe['fatal_n']} = {safe['fatal_rate']:.1%}** |
| Recall pangan | {safe['recall_pangan']:.1%} |
| Alarm beracun palsu pada pangan | {safe['alarm_palsu_beracun_pada_pangan']:.1%} |"""),
    ("Kalau ragu jangan dijawab", f"""| Cakupan | n | Akurasi |
|---|---|---|
{tbl_abs}"""),
    ("15 kelas terburuk", f"""| Spesies | Recall | n |
|---|---|---|
{tbl_worst}"""),
    ("10 kelas terbaik", f"""| Spesies | Recall |
|---|---|
{tbl_best}"""),
    ("Cara membaca", """- **Top-1**: tebakan peringkat 1 benar.
- **Top-3**: jawaban benar ada di 3 kandidat teratas (jaring pengaman aplikasi).
- **Kesalahan fatal**: foto tanaman beracun ditebak sebagai spesies yang aman dimakan.
  Ini metrik paling penting untuk aplikasi keselamatan lapangan."""),
]

In [30]:
write_md(D / "HASIL.md", "Model 1 - CNN MobileNetV2", sections)
print("DONE MODEL 1", flush=True)

  md -> C:\Users\Wahyu Pratomo\Project Machine Learning\tanaman-obat\output\01_cnn\HASIL.md
DONE MODEL 1


---

# Tahap 3 · Model 2 — SVM RBF di atas embedding CNN

SVM tidak melihat gambar, melainkan vektor 1.280 dimensi keluaran
lapisan `embedding` model 1. Standarisasi + kernel RBF, `C=10`, 100 kelas
one-vs-rest.

Probabilitas di sini softmax bertemperatur (T=10) di atas `decision_function`
— hanya agar kurva abstainnya sebanding dengan model lain.

**Keluar:** `output/02_svm/`.

In [31]:
import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

sys.path.insert(0, r"C:\Users\Wahyu Pratomo\Project Machine Learning\tanaman-obat\scripts")
from common import (OUT, abstain_curve, load_labels, load_tags, safety_metrics,  # noqa: E402
                    save_json, topk_acc, worst_classes, write_md)

import matplotlib  # noqa: E402
matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402

SRC = OUT / "01_cnn"
D = OUT / "02_svm"
FIG = D / "figures"
FIG.mkdir(parents=True, exist_ok=True)

In [32]:
labels = load_labels()
N = len(labels)
Xtr, Xva, Xte = (np.load(SRC / "emb_train.npy"), np.load(SRC / "emb_val.npy"),
                 np.load(SRC / "emb_test.npy"))
ytr, yva, yte = (np.load(SRC / "y_train.npy"), np.load(SRC / "y_val.npy"),
                 np.load(SRC / "y_test.npy"))
print(f"embedding train {Xtr.shape} test {Xte.shape} | kelas {N}", flush=True)

embedding train (6887, 1280) test (1476, 1280) | kelas 100


In [33]:
t0 = time.time()
pipe = make_pipeline(StandardScaler(),
                     SVC(C=10, gamma="scale", kernel="rbf",
                         decision_function_shape="ovr", cache_size=3000))
pipe.fit(Xtr, ytr)
train_sec = time.time() - t0
print(f"fit SVM: {train_sec/60:.1f} menit", flush=True)

fit SVM: 0.9 menit


In [34]:
score = pipe.decision_function(Xte)
pred = score.argmax(1)

# skor -> pseudo-probabilitas (softmax bertemperatur) supaya kurva abstain sebanding

In [35]:
def to_prob(s, T=10.0):
    z = s / T
    z -= z.max(1, keepdims=True)
    p = np.exp(z)
    return p / p.sum(1, keepdims=True)

In [36]:
probs = to_prob(score)
top1 = accuracy_score(yte, pred)
top3 = topk_acc(yte, probs, 3)
macro = f1_score(yte, pred, average="macro")

In [37]:
np.save(D / "skor_keputusan_test.npy", score)
np.save(D / "probabilitas_test.npy", probs)
np.save(D / "pred_test.npy", pred)

In [38]:
tags = load_tags()
safe, fatal = safety_metrics(yte, pred, labels, tags)
metrics = {"model": "SVM RBF (embedding CNN)", "top1": float(top1), "top3": float(top3),
           "macro_f1": float(macro), "train_sec": round(train_sec, 1), **safe}

In [39]:
save_json(D / "metrics.json", metrics)
print(f"TOPI-1 {top1:.4f} | TOP-3 {top3:.4f} | macroF1 {macro:.4f}", flush=True)

cm = confusion_matrix(yte, pred, labels=list(range(N)))

  json -> C:\Users\Wahyu Pratomo\Project Machine Learning\tanaman-obat\output\02_svm\metrics.json
TOPI-1 0.8991 | TOP-3 0.9566 | macroF1 0.8966


In [40]:
fig, ax = plt.subplots(figsize=(12, 11))
ax.imshow(cm, cmap="magma")
ax.set_title("Confusion matrix SVM RBF (100 kelas)"); ax.set_xlabel("prediksi"); ax.set_ylabel("aktual")
plt.tight_layout(); plt.savefig(FIG / "confusion_matrix.png", dpi=110); plt.close()

In [41]:
per_class = cm.diagonal() / np.maximum(cm.sum(1), 1)
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(len(per_class)), per_class[np.argsort(per_class)], color="#e65100")
ax.set_title("Recall per kelas (urut naik) - SVM"); ax.grid(axis="y", alpha=.3)
plt.tight_layout(); plt.savefig(FIG / "recall_per_kelas.png", dpi=120); plt.close()

In [42]:
cur = abstain_curve(yte, pred, probs.max(1))
cur.to_csv(D / "kurva_abstain.csv", index=False)
fig, ax = plt.subplots(figsize=(7, 4.4))
ax.plot(cur.coverage * 100, cur.accuracy * 100, marker="o", color="#e65100")
ax.set_xlabel("Foto yang dijawab (%)"); ax.set_ylabel("Akurasi pada yang dijawab (%)")
ax.set_title("SVM: kalau ragu jangan dijawab"); ax.invert_xaxis(); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig(FIG / "kurva_abstain.png", dpi=130); plt.close()

In [43]:
wk, best = worst_classes(yte, pred, labels, 15)
wk.to_csv(D / "kelas_terburuk.csv", index=False)

tbl_abs = "\n".join(f"| {r.coverage:.0%} | {r.n} | {r.accuracy:.1%} |" for r in cur.itertuples())
tbl_worst = "\n".join(f"| {r.spesies} | {r.recall:.2f} | {r.n} |" for r in wk.itertuples())

In [44]:
sections = [
    ("Ringkasan", f"""| Metrik | Nilai |
|---|---|
| Top-1 | **{top1:.1%}** |
| Top-3 | **{top3:.1%}** |
| Macro-F1 | {macro:.1%} |
| Waktu fit | {train_sec/60:.1f} menit |
| Input | embedding 1.280 dimensi dari model 1 |"""),
    ("Keamanan", f"""| Metrik | Nilai |
|---|---|
| Foto beracun di test | {safe['n_test_beracun']} |
| Dikenali sebagai beracun | {safe['recall_beracun_sebagai_beracun']:.1%} |
| **Kesalahan fatal** | **{safe['fatal_n']} = {safe['fatal_rate']:.1%}** |
| Recall pangan | {safe['recall_pangan']:.1%} |
| Alarm beracun palsu pada pangan | {safe['alarm_palsu_beracun_pada_pangan']:.1%} |"""),
    ("Kalau ragu jangan dijawab", f"""| Cakupan | n | Akurasi |
|---|---|---|
{tbl_abs}"""),
    ("15 kelas terburuk", f"""| Spesies | Recall | n |
|---|---|---|
{tbl_worst}"""),
    ("Catatan", """SVM memakai embedding dari model 1, bukan gambar langsung.
Kalau model 1 lemah, SVM tidak bisa lebih pintar dari fitur yang diberi.
Probabilitas di sini adalah softmax bertemperatur (T=10) di atas decision_function,
hanya untuk membuat kurva abstain sebanding dengan model lain."""),
]

In [45]:
write_md(D / "HASIL.md", "Model 2 - SVM RBF", sections)
print("DONE MODEL 2", flush=True)

  md -> C:\Users\Wahyu Pratomo\Project Machine Learning\tanaman-obat\output\02_svm\HASIL.md
DONE MODEL 2


---

# Tahap 4 · Model 3 — Random Forest 300 pohon

Random Forest di atas embedding yang sama,
`class_weight=balanced_subsample` untuk kelas tidak seimbang.

**Keluar:** `output/03_rf/`.

In [46]:
import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score

sys.path.insert(0, r"C:\Users\Wahyu Pratomo\Project Machine Learning\tanaman-obat\scripts")
from common import (OUT, abstain_curve, load_labels, load_tags, safety_metrics,  # noqa: E402
                    save_json, topk_acc, worst_classes, write_md)

import matplotlib  # noqa: E402
matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402

SRC = OUT / "01_cnn"
D = OUT / "03_rf"
FIG = D / "figures"
FIG.mkdir(parents=True, exist_ok=True)

In [47]:
labels = load_labels()
N = len(labels)
Xtr = np.load(SRC / "emb_train.npy")
Xte = np.load(SRC / "emb_test.npy")
ytr = np.load(SRC / "y_train.npy")
yte = np.load(SRC / "y_test.npy")
print(f"embedding train {Xtr.shape} test {Xte.shape} | kelas {N}", flush=True)

embedding train (6887, 1280) test (1476, 1280) | kelas 100


In [48]:
t0 = time.time()
rf = RandomForestClassifier(n_estimators=300, max_features="sqrt", n_jobs=-1,
                            random_state=42, class_weight="balanced_subsample")
rf.fit(Xtr, ytr)
train_sec = time.time() - t0
print(f"fit RF: {train_sec/60:.1f} menit", flush=True)

fit RF: 0.7 menit


In [49]:
probs = rf.predict_proba(Xte)
pred = probs.argmax(1)
top1 = accuracy_score(yte, pred)
top3 = topk_acc(yte, probs, 3)
macro = f1_score(yte, pred, average="macro")

In [50]:
np.save(D / "probabilitas_test.npy", probs)
np.save(D / "pred_test.npy", pred)

tags = load_tags()
safe, fatal = safety_metrics(yte, pred, labels, tags)

In [51]:
metrics = {"model": "Random Forest 300", "top1": float(top1), "top3": float(top3),
           "macro_f1": float(macro), "train_sec": round(train_sec, 1), **safe}
save_json(D / "metrics.json", metrics)
print(f"TOPI-1 {top1:.4f} | TOP-3 {top3:.4f} | macroF1 {macro:.4f}", flush=True)

  json -> C:\Users\Wahyu Pratomo\Project Machine Learning\tanaman-obat\output\03_rf\metrics.json
TOPI-1 0.8720 | TOP-3 0.9533 | macroF1 0.8660


In [52]:
cm = confusion_matrix(yte, pred, labels=list(range(N)))
fig, ax = plt.subplots(figsize=(12, 11))
ax.imshow(cm, cmap="cividis")
ax.set_title("Confusion matrix Random Forest (100 kelas)")
ax.set_xlabel("prediksi"); ax.set_ylabel("aktual")
plt.tight_layout(); plt.savefig(FIG / "confusion_matrix.png", dpi=110); plt.close()

In [53]:
per_class = cm.diagonal() / np.maximum(cm.sum(1), 1)
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(len(per_class)), per_class[np.argsort(per_class)], color="#6a1b9a")
ax.set_title("Recall per kelas (urut naik) - Random Forest"); ax.grid(axis="y", alpha=.3)
plt.tight_layout(); plt.savefig(FIG / "recall_per_kelas.png", dpi=120); plt.close()

imp = pd.Series(rf.feature_importances_).sort_values(ascending=False)

In [54]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(30), imp.values[:30], color="#6a1b9a")
ax.set_title("30 fitur embedding paling penting"); ax.set_xlabel("peringkat fitur")
plt.tight_layout(); plt.savefig(FIG / "fitur_penting.png", dpi=120); plt.close()

In [55]:
cur = abstain_curve(yte, pred, probs.max(1))
cur.to_csv(D / "kurva_abstain.csv", index=False)
fig, ax = plt.subplots(figsize=(7, 4.4))
ax.plot(cur.coverage * 100, cur.accuracy * 100, marker="o", color="#6a1b9a")
ax.set_xlabel("Foto yang dijawab (%)"); ax.set_ylabel("Akurasi pada yang dijawab (%)")
ax.set_title("Random Forest: kalau ragu jangan dijawab"); ax.invert_xaxis(); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig(FIG / "kurva_abstain.png", dpi=130); plt.close()

In [56]:
wk, best = worst_classes(yte, pred, labels, 15)
wk.to_csv(D / "kelas_terburuk.csv", index=False)

tbl_abs = "\n".join(f"| {r.coverage:.0%} | {r.n} | {r.accuracy:.1%} |" for r in cur.itertuples())
tbl_worst = "\n".join(f"| {r.spesies} | {r.recall:.2f} | {r.n} |" for r in wk.itertuples())

In [57]:
sections = [
    ("Ringkasan", f"""| Metrik | Nilai |
|---|---|
| Top-1 | **{top1:.1%}** |
| Top-3 | **{top3:.1%}** |
| Macro-F1 | {macro:.1%} |
| Waktu fit | {train_sec/60:.1f} menit |
| Jumlah pohon | 300 |"""),
    ("Keamanan", f"""| Metrik | Nilai |
|---|---|
| Foto beracun di test | {safe['n_test_beracun']} |
| Dikenali sebagai beracun | {safe['recall_beracun_sebagai_beracun']:.1%} |
| **Kesalahan fatal** | **{safe['fatal_n']} = {safe['fatal_rate']:.1%}** |
| Recall pangan | {safe['recall_pangan']:.1%} |
| Alarm beracun palsu pada pangan | {safe['alarm_palsu_beracun_pada_pangan']:.1%} |"""),
    ("Kalau ragu jangan dijawab", f"""| Cakupan | n | Akurasi |
|---|---|---|
{tbl_abs}"""),
    ("15 kelas terburuk", f"""| Spesies | Recall | n |
|---|---|---|
{tbl_worst}"""),
    ("Catatan", """Random Forest memakai embedding dari model 1.
Kelebihannya: tahan noise label. Kekurangannya: biasanya kalah akurat dari CNN
karena tidak bisa memakai struktur spasial gambar."""),
]

In [58]:
write_md(D / "HASIL.md", "Model 3 - Random Forest 300", sections)
print("DONE MODEL 3", flush=True)

  md -> C:\Users\Wahyu Pratomo\Project Machine Learning\tanaman-obat\output\03_rf\HASIL.md
DONE MODEL 3


---

# Tahap 5 · Perbandingan 3 Algoritma

Menyatukan ketiga model di data test yang sama: top-1, top-3,
macro-F1, kesalahan fatal, alarm palsu, kurva akurasi-vs-cakupan, daftar
pasangan fatal.

**Keluar:** `output/04_perbandingan/` → `tabel_model.csv`, `PERBANDINGAN.md`,
`kesalahan_fatal.csv`, 3 grafik. Tidak ada training di sini.

In [59]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, r"C:\Users\Wahyu Pratomo\Project Machine Learning\tanaman-obat\scripts")
from common import OUT, load_labels, load_tags, safety_metrics, save_json, write_md  # noqa: E402

import matplotlib  # noqa: E402
matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
from sklearn.metrics import accuracy_score, f1_score  # noqa: E402

D = OUT / "04_perbandingan"
FIG = D / "figures"
FIG.mkdir(parents=True, exist_ok=True)

In [60]:
labels = load_labels()
tags = load_tags()
yte = np.load(OUT / "01_cnn" / "y_test.npy")

# cakupan dari aturan penolakan (kalau sudah ada) supaya semua model bisa
# dibandingkan pada jumlah foto yang dijawab sama
_rules_f = OUT / "05_bahan_aplikasi" / "safety_rules.json"
RULES = (json.loads(_rules_f.read_text(encoding="utf-8")) if _rules_f.exists() else None)
COV = RULES["test_pada_ambang_ini"]["cakupan"] if RULES else None

models = {
    "CNN MobileNetV2": ("01_cnn", "probabilitas_test.npy"),
    "SVM RBF": ("02_svm", "probabilitas_test.npy"),
    "Random Forest 300": ("03_rf", "probabilitas_test.npy"),
}

In [61]:
rows, curves, percat = [], {}, []
fatal_rows = []
for name, (folder, fname) in models.items():
    p = np.load(OUT / folder / fname)
    pred = p.argmax(1)
    conf = p.max(1)
    safe, fatal = safety_metrics(yte, pred, labels, tags)
    acc_cov = None
    if COV:
        k = int(round(COV * len(yte)))
        sel = np.argsort(-conf)[:k]
        acc_cov = float((pred[sel] == yte[sel]).mean())
    rows.append({
        "model": name,
        "top1": accuracy_score(yte, pred),
        "macro_f1": f1_score(yte, pred, average="macro"),
        "fatal_rate": safe["fatal_rate"],
        "fatal_n": safe["fatal_n"],
        "recall_beracun": safe["recall_beracun_sebagai_beracun"],
        "false_alarm_beracun": safe["alarm_palsu_beracun_pada_pangan"],
        "akurasi_pada_cakupan": acc_cov,
    })
    percat.append({"model": name, **{k: safe[k] for k in
                                     ("recall_pangan", "recall_beracun_sebagai_beracun",
                                      "alarm_palsu_beracun_pada_pangan", "fatal_rate")}})
    cur = pd.read_csv(OUT / folder / "kurva_abstain.csv")
    curves[name] = cur
    for lbl, pr in zip(labels, (pred == yte)):
        pass
    # kesalahan fatal rinci
    tox = np.array([bool(tags.loc[l, "beracun"]) for l in labels])
    food = np.array([bool(tags.loc[l, "pangan"]) for l in labels])
    for i in np.where(fatal)[0]:
        fatal_rows.append({"model": name,
                           "aktual": labels[yte[i]],
                           "ditebak": labels[pred[i]]})

In [62]:
comp = pd.DataFrame(rows)
# top-3 dihitung ulang seragam
t3 = []
for name, (folder, fname) in models.items():
    p = np.load(OUT / folder / fname)
    top = np.argsort(-p, axis=1)[:, :3]
    t3.append(float(np.mean([yte[i] in top[i] for i in range(len(yte))])))
comp["top3"] = t3
comp = comp[["model", "top1", "top3", "macro_f1", "recall_beracun",
             "fatal_rate", "fatal_n", "false_alarm_beracun",
             "akurasi_pada_cakupan"]]

In [63]:
comp.to_csv(D / "tabel_model.csv", index=False)
pd.DataFrame(percat).to_csv(D / "per_kategori.csv", index=False)
pd.DataFrame(fatal_rows).to_csv(D / "kesalahan_fatal.csv", index=False)
print(comp.to_string(index=False), flush=True)

            model     top1     top3  macro_f1  recall_beracun  fatal_rate  fatal_n  false_alarm_beracun  akurasi_pada_cakupan
  CNN MobileNetV2 0.891599 0.952575  0.888104        0.923423    0.042793       19             0.139257              0.927431
          SVM RBF 0.899051 0.956640  0.896595        0.927928    0.036036       16             0.136605              0.905660
Random Forest 300 0.871951 0.953252  0.865987        0.916667    0.042793       19             0.141910              0.907112


In [64]:
# --- figur: bar akurasi ---
fig, ax = plt.subplots(figsize=(8, 4.6))
x = np.arange(len(comp))
ax.bar(x - 0.2, comp.top1 * 100, width=0.38, label="Top-1", color="#1565c0")
ax.bar(x + 0.2, comp.top3 * 100, width=0.38, label="Top-3", color="#66bb6a")
for i, (a, b) in enumerate(zip(comp.top1, comp.top3)):
    ax.text(i - 0.2, a * 100 + 0.6, f"{a*100:.1f}", ha="center", fontsize=9)
    ax.text(i + 0.2, b * 100 + 0.6, f"{b*100:.1f}", ha="center", fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(comp.model, fontsize=9)
ax.set_ylabel("Akurasi (%)"); ax.set_ylim(0, 113)
ax.set_title("Perbandingan akurasi 3 algoritma")
ax.legend(loc="upper center", ncol=2, frameon=False)
ax.grid(axis="y", alpha=.3)
plt.tight_layout(); plt.savefig(FIG / "bar_akurasi.png", dpi=130); plt.close()

In [65]:
# --- kurva abstain gabungan ---
fig, ax = plt.subplots(figsize=(7.4, 4.6))
for name, cur in curves.items():
    ax.plot(cur.coverage * 100, cur.accuracy * 100, marker="o", label=name)
ax.set_xlabel("Foto yang dijawab (%)"); ax.set_ylabel("Akurasi pada yang dijawab (%)")
ax.set_title("Kalau model ragu, jangan dijawab")
ax.invert_xaxis(); ax.grid(alpha=.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(FIG / "kurva_abstain_semua.png", dpi=130); plt.close()

In [66]:
# --- figur fatal ---
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.bar(comp.model, comp.fatal_rate * 100, color="#c62828")
for i, v in enumerate(comp.fatal_rate * 100):
    ax.text(i, v + 0.15, f"{v:.1f}%", ha="center", fontsize=9)
ax.set_ylabel("Kesalahan fatal (%)")
ax.set_ylim(0, float(comp.fatal_rate.max()) * 100 * 1.35)
ax.set_title("Beracun ditebak sebagai tanaman pangan")
ax.grid(axis="y", alpha=.3)
plt.tight_layout(); plt.savefig(FIG / "kesalahan_fatal.png", dpi=130); plt.close()

cur_all = pd.concat([c.assign(model=k) for k, c in curves.items()])
cur_all.to_csv(D / "kurva_abstain.csv", index=False)

In [67]:
best = comp.sort_values("top1", ascending=False).iloc[0]
row = {r.model: r for r in comp.itertuples()}
cnn = row["CNN MobileNetV2"]
svm = row["SVM RBF"]
rfm = row["Random Forest 300"]
selisih = (svm.top1 - cnn.top1) * 100

In [68]:
def idn(v, d=1):
    return f"{v:.{d}f}".replace(".", ",")


# angka ambang + ukuran diambil dari artefak, bukan ditulis tangan
rules_f = OUT / "05_bahan_aplikasi" / "safety_rules.json"
meta_f = OUT / "05_bahan_aplikasi" / "model_meta.json"
if rules_f.exists():
    _r = json.loads(rules_f.read_text(encoding="utf-8"))
    ambang = _r["ambang_keyakinan"]
    tst = _r["test_pada_ambang_ini"]
    kal_ambang = (f"Dengan `AMBANG = {idn(ambang, 2)}`, "
                  f"{idn(tst['cakupan'] * 100)}% foto dijawab "
                  f"dengan akurasi {idn(tst['akurasi_pada_yang_dijawab'] * 100)}%")
else:
    kal_ambang = "Angka ambang belum dihitung (jalankan `06_threshold.py`)"
ukuran_tflite = (f"{idn(json.loads(meta_f.read_text(encoding='utf-8'))['size_mb'], 2)} MB"
                 if meta_f.exists() else "4,6 MB")

# perbandingan adil: pada cakupan yang sama, siapa paling akurat?
if COV and cnn.akurasi_pada_cakupan is not None:
    kal_cov = (f"Pada cakupan yang sama ({idn(COV * 100)}% foto dijawab), CNN justru "
               f"**paling akurat**: {idn(cnn.akurasi_pada_cakupan * 100)}% "
               f"vs SVM {idn(svm.akurasi_pada_cakupan * 100)}% "
               f"dan RF {idn(rfm.akurasi_pada_cakupan * 100)}%.")
    kal_menang = ("CNN menang setelah aturan penolakan dipakai, jadi SVM tidak "
                  "perlu dibawa ke HP sama sekali.")
    head_cov = f"Akurasi @ cakupan {idn(COV * 100)}%"
    acc_cnn = f"{idn(cnn.akurasi_pada_cakupan * 100)}%"
    acc_svm = f"{idn(svm.akurasi_pada_cakupan * 100)}%"
    acc_rfm = f"{idn(rfm.akurasi_pada_cakupan * 100)}%"
else:
    kal_cov = ("Perbandingan pada cakupan sama belum bisa dihitung karena "
               "`safety_rules.json` belum ada.")
    kal_menang = ""
    head_cov = "Akurasi @ cakupan"
    acc_cnn = acc_svm = acc_rfm = "—"

tbl_pil = "\n".join([
    f"| CNN murni (`model.tflite`) | {idn(cnn.top1 * 100)}% | {acc_cnn} | "
    f"{ukuran_tflite} | rendah, 1 inferensi |",
    f"| CNN + SVM | {idn(svm.top1 * 100)}% | {acc_svm} | belasan MB | "
    f"tinggi, kernel RBF manual |",
    f"| CNN + Random Forest | {idn(rfm.top1 * 100)}% | {acc_rfm} | ± 171 MB | "
    f"sangat tinggi, tidak layak |",
])

In [69]:
tbl = "\n".join(
    f"| {r.model} | {r.top1:.1%} | {r.top3:.1%} | {r.macro_f1:.1%} | "
    f"{r.fatal_rate:.1%} | {r.false_alarm_beracun:.1%} |" for r in comp.itertuples())

In [70]:
tbl_cov = ""
for name, c in curves.items():
    tbl_cov += f"\n**{name}**\n\n| Cakupan | Akurasi |\n|---|---|\n" + \
               "\n".join(f"| {r.coverage:.0%} | {r.accuracy:.1%} |" for r in c.itertuples()) + "\n"
tbl_fatal = "\n".join(f"| {r.model} | {r.aktual} | {r.ditebak} |"
                      for r in pd.DataFrame(fatal_rows).head(25).itertuples()) if fatal_rows else "(tidak ada)"

In [71]:
sections = [
    ("Tabel perbandingan", f"""| Model | Top-1 | Top-3 | Macro-F1 | Fatal | Alarm palsu |
|---|---|---|---|---|---|
{tbl}"""),
    ("Pemenang", f"""Kalau semua foto wajib dijawab, **{best.model}** unggul di top-1
({idn(best.top1 * 100)}%). Tapi aplikasi ini punya aturan "kalau ragu jangan
dijawab", dan aturan itu mengubah pemenangnya.

{kal_cov} {kal_menang}

Tabel di bawah memakai angka yang benar-benar dipakai aplikasi. Model 2 dan 3
butuh **dua tahap** di HP karena memakai embedding dari model 1 (jaringan CNN
+ perhitungan SVM/RF di Dart). Lihat `02_svm/DEPLOY.md` untuk hitungan
ukuran dan bebannya.

| Pilihan | Top-1 semua foto | {head_cov} | Ukuran tambahan | Kerumitan di aplikasi |
|---|---|---|---|
{tbl_pil}

**Rekomendasi paket ini: CNN murni.** {kal_ambang}, tanpa tambahan ukuran
maupun kode di HP."""),
    ("Akurasi vs cakupan", f"""Semakin banyak foto ditolak, semakin tinggi akurasi.
Inilah dasar aturan "kalau ragu jangan dijawab".
{tbl_cov}"""),
    ("Contoh kesalahan fatal", f"""| Model | Aktual (beracun) | Ditebak (aman dimakan) |
|---|---|---|
{tbl_fatal}"""),
    ("Cara membaca", """- **Top-1**: tebakan pertama benar.
- **Top-3**: jawaban benar ada di 3 kandidat teratas.
- **Fatal**: foto tanaman beracun ditebak sebagai spesies pangan yang aman.
  Ini metrik utama untuk keselamatan.
- **Alarm palsu**: tanaman pangan yang ditandai beracun, membuat pengguna panik."""),
]

In [72]:
write_md(D / "PERBANDINGAN.md", "Perbandingan 3 Algoritma", sections)
save_json(D / "ringkasan.json", {"tabel": comp.to_dict("records"),
                                 "pemenang": best.model})
print("DONE TAHAP 5", flush=True)

  md -> C:\Users\Wahyu Pratomo\Project Machine Learning\tanaman-obat\output\04_perbandingan\PERBANDINGAN.md
  json -> C:\Users\Wahyu Pratomo\Project Machine Learning\tanaman-obat\output\04_perbandingan\ringkasan.json
DONE TAHAP 5


---

# Tahap 6 · Export TFLite float16 + verifikasi

Konversi model 1 ke `.tflite` dengan bobot float16, lalu
**verifikasi numerik**: 200 gambar test dijalankan di Keras dan di TFLite,
argmax-nya dibandingkan.

**Keluar:** `output/05_bahan_aplikasi/` → `model.tflite`, `labels.json`,
`model_meta.json`, `metrics_model_1.json`, `README_MODEL.md`.

In [73]:
import json
import shutil
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf

sys.path.insert(0, r"C:\Users\Wahyu Pratomo\Project Machine Learning\tanaman-obat\scripts")
from common import DATA, OUT, save_json, write_md  # noqa: E402

SRC = OUT / "01_cnn"
D = OUT / "05_bahan_aplikasi"
D.mkdir(parents=True, exist_ok=True)

model = tf.keras.models.load_model(SRC / "model.keras")
IMG = int(model.input_shape[1])

In [74]:
labels = json.loads((DATA / "labels.json").read_text(encoding="utf-8"))
print(f"model dimuat | input {model.input_shape} | kelas {len(labels)}", flush=True)

# ---------- konversi ----------
conv = tf.lite.TFLiteConverter.from_keras_model(model)
conv.optimizations = [tf.lite.Optimize.DEFAULT]
conv.target_spec.supported_types = [tf.float16]
tflite_bytes = conv.convert()
(D / "model.tflite").write_bytes(tflite_bytes)
size_mb = len(tflite_bytes) / 1024 ** 2
print(f"TFLite float16: {size_mb:.2f} MB", flush=True)

model dimuat | input (None, 224, 224, 3) | kelas 100


INFO:tensorflow:Assets written to: C:\Users\WAHYUP~1\AppData\Local\Temp\tmp8ex8m92s\assets


INFO:tensorflow:Assets written to: C:\Users\WAHYUP~1\AppData\Local\Temp\tmp8ex8m92s\assets



Output Type:


TFLite float16: 4.56 MB


In [75]:
# ---------- verifikasi ----------
interp = tf.lite.Interpreter(model_path=str(D / "model.tflite"))
interp.allocate_tensors()
inp_det = interp.get_input_details()[0]
out_det = interp.get_output_details()[0]
print(f"input {inp_det['shape']} {inp_det['dtype']} | output {out_det['shape']}", flush=True)

te = pd.read_csv(DATA / "test.csv")
rng = np.random.default_rng(42)
sample = te.iloc[rng.choice(len(te), size=200, replace=False)]

input [  1 224 224   3] <class 'numpy.float32'> | output [  1 100]


In [76]:
def load_raw(p):
    raw = tf.io.read_file(str(p))
    img = tf.io.decode_image(raw, channels=3, expand_animations=False)
    img = tf.image.resize(img, [IMG, IMG])
    return tf.cast(img, tf.float32).numpy()[None, ...]


keras_probs, lite_probs = [], []
for p in sample.path.values:
    x = load_raw(p)
    kp = model.predict(x, verbose=0)[0]
    interp.set_tensor(inp_det["index"], x)
    interp.invoke()
    lp = interp.get_tensor(out_det["index"])[0]
    keras_probs.append(kp)
    lite_probs.append(lp)

kp = np.array(keras_probs)
lp = np.array(lite_probs)
agree = float((kp.argmax(1) == lp.argmax(1)).mean())
maxdiff = float(np.abs(kp - lp).max())
meandiff = float(np.abs(kp - lp).mean())
print(f"kesepakatan argmax: {agree:.3%} | max diff {maxdiff:.5f} | mean diff {meandiff:.6f}", flush=True)

y = sample.y.values
acc_keras = float((kp.argmax(1) == y).mean())
acc_lite = float((lp.argmax(1) == y).mean())
print(f"akurasi subset 200 gambar: keras {acc_keras:.3%} | tflite {acc_lite:.3%}", flush=True)

kesepakatan argmax: 100.000% | max diff 0.02470 | mean diff 0.000082


akurasi subset 200 gambar: keras 90.000% | tflite 90.000%


In [77]:
# ---------- aset untuk teman ----------
shutil.copy(DATA / "labels.json", D / "labels.json")
shutil.copy(SRC / "metrics.json", D / "metrics_model_1.json")

meta = {
    "input": {"shape": [1, IMG, IMG, 3], "dtype": "float32",
              "range": "0..255 (jangan dinormalisasi lagi, preprocessing ada di dalam model)",
              "color": "RGB"},
    "output": {"shape": [1, len(labels)], "dtype": "float32", "activation": "softmax",
               "meaning": "probabilitas per kelas, urutan sesuai labels.json"},
    "quantization": "float16 weights",
    "size_mb": round(size_mb, 2),
    "verification": {"n_images": len(sample), "argmax_agreement": agree,
                     "max_abs_diff": maxdiff, "mean_abs_diff": meandiff,
                     "acc_keras_subset": acc_keras, "acc_tflite_subset": acc_lite},
    "labels_count": len(labels),
}

In [78]:
save_json(D / "model_meta.json", meta)

sections = [
    ("Isi folder", f"""| File | Isi |
|---|---|
| `model.tflite` | Model siap pakai, input float32 0..255 ukuran 1x{IMG}x{IMG}x3 |
| `labels.json` | 100 nama kelas, urutan sama dengan output model |
| `model_meta.json` | Spesifikasi input/output + hasil verifikasi |
| `metrics_model_1.json` | Angka akurasi model |
| `safety_rules.json` | Nilai AMBANG + aturan penolakan |"""),
    ("Cara pakai di Flutter", f"""Tambahkan `tflite_flutter` di `pubspec.yaml`, taruh `model.tflite` dan
`labels.json` di folder `assets/`.

Alur inferensi:

1. Ambil gambar (kamera/galeri), decode jadi RGB.
2. Resize ke **{IMG}x{IMG}** (bilinear).
3. Konversi ke float32 dengan nilai **0..255** — JANGAN dibagi 255.
   Normalisasi sudah ada di dalam model (`Rescaling(1/127.5, offset=-1)`).
4. Bentuk tensor `[1, {IMG}, {IMG}, 3]`, jalankan interpreter.
5. Output `[1, 100]` = probabilitas, urutannya sesuai `labels.json`.
6. Ambil 3 nilai terbesar -> top-3 kandidat.
7. Tolak menjawab kalau probabilitas tertinggi < `AMBANG` (lihat `safety_rules.json`)."""),
    ("Verifikasi", f"""| Pemeriksaan | Hasil |
|---|---|
| Kesepakatan argmax Keras vs TFLite (200 gambar) | {agree:.2%} |
| Selisih probabilitas maksimum | {maxdiff:.5f} |
| Ukuran model | {size_mb:.2f} MB |

Kalau teman Anda mendapat hasil berbeda jauh di HP, kemungkinan besar
preprocessing-nya tidak sama (resize atau rentang nilai)."""),
    ("Catatan penting", """- Model ini **hanya** mengenali 100 spesies di `labels.json`.
- Tanaman di luar daftar tetap akan dipaksa masuk ke salah satu kelas,
  karena itu aturan penolakan berbasis confidence WAJIB dipakai.
- Jangan menampilkan satu jawaban saja; selalu tampilkan top-3."""),
]

  json -> C:\Users\Wahyu Pratomo\Project Machine Learning\tanaman-obat\output\05_bahan_aplikasi\model_meta.json


In [79]:
write_md(D / "README_MODEL.md", "Model TFLite untuk Aplikasi", sections)
print("DONE TAHAP 6", flush=True)

  md -> C:\Users\Wahyu Pratomo\Project Machine Learning\tanaman-obat\output\05_bahan_aplikasi\README_MODEL.md
DONE TAHAP 6


---

# Tahap 6a · Menentukan AMBANG penolakan

Aturan "kalau ragu jangan dijawab" butuh satu angka. Ambang
dipilih dari **data validasi** (bukan test, supaya test tetap bersih): ambang
terendah yang masih memberi akurasi ≥ 93% pada foto yang dijawab, dengan
cakupan ≥ 40%.

**Keluar:** `output/05_bahan_aplikasi/safety_rules.json` +
`figures/ambang.png`.

In [80]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, r"C:\Users\Wahyu Pratomo\Project Machine Learning\tanaman-obat\scripts")
from common import OUT, load_tags, save_json

import matplotlib  # noqa: E402
matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402

SRC = OUT / "01_cnn"
D = OUT / "05_bahan_aplikasi"
FIG = D / "figures"
FIG.mkdir(parents=True, exist_ok=True)

TARGET = 0.93          # akurasi yang diinginkan pada foto yang dijawab
MIN_COVERAGE = 0.40    # jangan sampai aplikasi menolak > 60% foto

In [81]:
labels = json.loads((OUT / "_data" / "labels.json").read_text(encoding="utf-8"))
prob_va = np.load(SRC / "probabilitas_val.npy")
y_va = np.load(SRC / "y_val.npy")
conf = prob_va.max(1)
pred = prob_va.argmax(1)
correct = (pred == y_va)

print(f"validasi: {len(y_va)} gambar | akurasi penuh {(correct).mean():.4f}", flush=True)

validasi: 1476 gambar | akurasi penuh 0.8963


In [82]:
def stat(thr):
    sel = conf >= thr
    n = int(sel.sum())
    if n == 0:
        return {"ambang": float(thr), "n_dijawab": 0, "cakupan": 0.0, "akurasi": None}
    return {"ambang": float(thr), "n_dijawab": n, "cakupan": float(sel.mean()),
            "akurasi": float(correct[sel].mean())}

In [83]:
grid = np.round(np.arange(0.05, 0.96, 0.01), 2)
rows = [stat(t) for t in grid]
df = pd.DataFrame(rows)
df.to_csv(SRC / "kurva_ambang_val.csv", index=False)

In [84]:
ok = df[(df.akurasi.notna()) & (df.akurasi >= TARGET) & (df.cakupan >= MIN_COVERAGE)]
if len(ok):
    chosen = ok.sort_values("ambang").iloc[0]          # ambang terendah yang memenuhi
    note = f"ambang terendah yang memberi akurasi >= {TARGET:.0%} pada foto yang dijawab"
else:
    # tidak ada yang memenuhi -> pakai ambang dengan akurasi tertinggi, cakupan >= minimum
    cand = df[df.cakupan >= MIN_COVERAGE]
    chosen = cand.sort_values("akurasi", ascending=False).iloc[0]
    note = (f"tidak ada ambang yang mencapai {TARGET:.0%} dengan cakupan >= {MIN_COVERAGE:.0%}; "
            "dipakai ambang dengan akurasi terbaik")

AMBANG = float(chosen.ambang)
print(f"\nAMBANG = {AMBANG:.2f} | cakupan {chosen.cakupan:.1%} | "
      f"akurasi pada yang dijawab {chosen.akurasi:.1%}", flush=True)


AMBANG = 0.27 | cakupan 94.0% | akurasi pada yang dijawab 93.0%


In [85]:
# --- evaluasi ambang ini di TEST (sekali, hanya untuk laporan) ---
prob_te = np.load(SRC / "probabilitas_test.npy")
y_te = np.load(SRC / "y_test.npy")
conf_te = prob_te.max(1)
pred_te = prob_te.argmax(1)
sel = conf_te >= AMBANG
acc_te = float((pred_te[sel] == y_te[sel]).mean())

In [86]:
tags = load_tags()
tox = np.array([bool(tags.loc[l, "beracun"]) for l in labels])
food = np.array([bool(tags.loc[l, "pangan"]) for l in labels])
fatal_all = tox[y_te] & (pred_te != y_te) & food[pred_te] & ~tox[pred_te]
fatal_sel = fatal_all & sel
n_tox_sel = int((tox[y_te] & sel).sum())

print(f"TEST pada ambang ini: dijawab {sel.mean():.1%} | akurasi {acc_te:.1%} | "
      f"fatal {int(fatal_sel.sum())}/{n_tox_sel}", flush=True)

TEST pada ambang ini: dijawab 93.4% | akurasi 92.7% | fatal 9/410


In [87]:
rules = {
    "ambang_keyakinan": round(AMBANG, 2),
    "aturan": "tolak menjawab jika probabilitas tertinggi < ambang_keyakinan",
    "dasar_penentuan": note,
    "dipilih_dari": "data validasi (1.476 gambar), bukan data test",
    "target_akurasi_pada_yang_dijawab": TARGET,
    "cakupan_minimum": MIN_COVERAGE,
    "validasi": {k: (round(v, 4) if isinstance(v, float) else v) for k, v in chosen.to_dict().items()},
    "test_pada_ambang_ini": {
        "n_test": int(len(y_te)),
        "cakupan": round(float(sel.mean()), 4),
        "akurasi_pada_yang_dijawab": round(acc_te, 4),
        "akurasi_tanpa_penolakan": round(float((pred_te == y_te).mean()), 4),
        "fatal_n": int(fatal_sel.sum()),
        "fatal_rate_pada_beracun_yang_dijawab": round(float(fatal_sel.sum() / max(n_tox_sel, 1)), 4),
        "beracun_dijawab": n_tox_sel,
    },
    "aturan_margin": {
        "dipakai": False,
        "alasan": "sudah diuji pada model ini: aturan margin tidak mengubah cakupan maupun akurasi",
    },
    "larangan": [
        "jangan menurunkan ambang agar pengguna tidak kecewa",
        "jangan memakai satu jawaban saja; selalu tampilkan top-3",
        "jika salah satu dari top-3 punya level_racun >= 2, tampilkan peringatan merah",
    ],
}

In [88]:
save_json(D / "safety_rules.json", rules)

# --- figur ---
fig, ax = plt.subplots(figsize=(7.2, 4.6))
d = df[df.akurasi.notna()]
ax.plot(d.cakupan * 100, d.akurasi * 100, color="#1565c0", lw=1.6)
ax.axhline(TARGET * 100, ls="--", color="#c62828", lw=1, label=f"target {TARGET:.0%}")
ax.scatter([chosen.cakupan * 100], [chosen.akurasi * 100], s=70, color="#c62828", zorder=5,
           label=f"AMBANG {AMBANG:.2f}")
ax.set_xlabel("Foto yang dijawab (%)"); ax.set_ylabel("Akurasi pada yang dijawab (%)")
ax.set_title("Pemilihan AMBANG dari data validasi")
ax.invert_xaxis(); ax.grid(alpha=.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(FIG / "ambang.png", dpi=130); plt.close()

print("DONE TAHAP 6a", flush=True)

  json -> C:\Users\Wahyu Pratomo\Project Machine Learning\tanaman-obat\output\05_bahan_aplikasi\safety_rules.json


DONE TAHAP 6a


---

# Tahap 6b · Biaya deploy SVM di HP

Model paling akurat belum tentu model yang dikirim. Script ini
menghitung konsekuensi membawa SVM ke aplikasi: jumlah support vector, ukuran
bobot, jumlah dot product per inferensi.

**Keluar:** `output/02_svm/DEPLOY.md`.

In [89]:
import json
import sys
import time
from pathlib import Path

import numpy as np
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

sys.path.insert(0, r"C:\Users\Wahyu Pratomo\Project Machine Learning\tanaman-obat\scripts")
from common import OUT, write_md

SRC = OUT / "01_cnn"
D = OUT / "02_svm"

In [90]:
Xtr = np.load(SRC / "emb_train.npy")
ytr = np.load(SRC / "y_train.npy")
Xte = np.load(SRC / "emb_test.npy")
print(f"train {Xtr.shape}", flush=True)

train (6887, 1280)


In [91]:
t0 = time.time()
pipe = make_pipeline(StandardScaler(),
                     SVC(C=10, gamma="scale", kernel="rbf",
                         decision_function_shape="ovr", cache_size=3000))
pipe.fit(Xtr, ytr)
print(f"refit {time.time()-t0:.1f} s", flush=True)

svm = pipe[-1]
scaler = pipe[0]
n_sv = int(svm.n_support_.sum())
n_features = Xtr.shape[1]
n_class = len(svm.classes_)

sv_float32_mb = n_sv * n_features * 4 / 1024 ** 2
sv_float16_mb = n_sv * n_features * 2 / 1024 ** 2
dual_coef_mb = svm.dual_coef_.size * 4 / 1024 ** 2

# biaya inferensi SVM di HP: kernel RBF = 1 dot product per support vector
kernels = n_sv                       # jumlah evaluasi kernel
flops = kernels * n_features * 2      # kali + tambah
per_class_scores = n_class

refit 40.7 s


In [92]:
t0 = time.time()
_ = pipe.decision_function(Xte[:200])
lat_py = (time.time() - t0) / 200

print(f"support vector {n_sv:,} | float32 {sv_float32_mb:.1f} MB | "
      f"float16 {sv_float16_mb:.1f} MB", flush=True)
print(f"kernel evals per inferensi {kernels:,} | ~{flops/1e6:.0f} MFLOP", flush=True)
print(f"latency referensi (desktop python) {lat_py*1000:.1f} ms/gambar", flush=True)

rec = "CNN-embedding + SVM RBF"

support vector 5,468 | float32 26.7 MB | float16 13.3 MB


kernel evals per inferensi 5,468 | ~14 MFLOP


latency referensi (desktop python) 7.1 ms/gambar


In [93]:
def idn(v, d=1):
    return f"{v:.{d}f}".replace(".", ",")


# angka akurasi dibaca dari artefak, bukan ditulis tangan

In [94]:
def _read(p):
    return json.loads(Path(p).read_text(encoding="utf-8")) if Path(p).exists() else {}


m_svm = _read(D / "metrics.json")
m_cnn = _read(OUT / "01_cnn" / "metrics.json")

In [95]:
rules = _read(OUT / "05_bahan_aplikasi" / "safety_rules.json")
top1_svm = m_svm.get("top1")
top1_cnn = m_cnn.get("top1")
gain = (top1_svm - top1_cnn) * 100 if (top1_svm and top1_cnn) else None
kal_hasil = f"top-1 {idn(top1_svm*100)}%" if top1_svm else "akurasi tertinggi"

# akurasi pada cakupan yang sama, dari tabel perbandingan
acc_cov = {}
_tbl = OUT / "04_perbandingan" / "tabel_model.csv"
if _tbl.exists():
    import csv as _csv
    with _tbl.open(encoding="utf-8") as fh:
        for _row in _csv.DictReader(fh):
            _v = _row.get("akurasi_pada_cakupan")
            acc_cov[_row["model"]] = float(_v) if _v not in (None, "", "nan") else None
cnn_cov = acc_cov.get("CNN MobileNetV2")
svm_cov = acc_cov.get("SVM RBF")
cov = rules["test_pada_ambang_ini"]["cakupan"] if rules else None

if rules:
    _t = rules["test_pada_ambang_ini"]
    kal_tutup = (f"dengan `AMBANG = {idn(rules['ambang_keyakinan'], 2)}`, "
                 f"{idn(_t['cakupan'] * 100)}% foto dijawab dengan akurasi "
                 f"**{idn(_t['akurasi_pada_yang_dijawab']*100)}%**")
else:
    kal_tutup = "aturan penolakan"

if cnn_cov is not None and svm_cov is not None and cov:
    kal_banding = (f"Pada cakupan yang sama ({idn(cov * 100)}% foto dijawab), CNN "
                   f"malah **lebih akurat** daripada SVM: {idn(cnn_cov * 100)}% vs "
                   f"{idn(svm_cov * 100)}%. Jadi SVM tidak lagi punya alasan untuk "
                   f"dibawa ke HP.")
    kal_gain = (f"Kalau semua foto wajib dijawab, akurasi SVM naik {idn(gain)} poin "
                f"dibanding CNN murni — tapi keunggulan itu hilang begitu aturan "
                f"penolakan dipakai." if gain is not None else "")
else:
    kal_banding = "Perbandingan pada cakupan yang sama ada di `04_perbandingan/PERBANDINGAN.md`."
    kal_gain = f"Akurasi naik {idn(gain)} poin dibanding CNN murni." if gain is not None else ""

In [96]:
body = f"""SVM adalah model dengan akurasi tertinggi ({kal_hasil}), **tetapi SVM
tidak bisa berdiri sendiri**. Dia bekerja di atas vektor 1.280 dimensi yang
dihasilkan MobileNetV2. Artinya di HP harus ada dua tahap:

1. Gambar -> TFLite MobileNetV2 (sampai lapisan `embedding`) -> 1.280 angka
2. 1.280 angka -> hitung kernel RBF ke semua support vector -> pilih kelas

| Item | Nilai |
|---|---|
| Support vector | {n_sv:,} |
| Dimensi fitur | {n_features} |
| Ukuran bobot SVM (float32) | **{idn(sv_float32_mb)} MB** |
| Ukuran bobot SVM (float16) | {idn(sv_float16_mb)} MB |
| Ukuran backbone CNN (tanpa kepala) | ~3,7 MB |
| Evaluasi kernel per inferensi | {n_sv:,} x {n_features} |
| Beban tambahan per inferensi | ~{flops/1e6:.0f} MFLOP, {lat_py*1000:.0f} ms di laptop ini |

Konsekuensi praktis:

- **Ukuran jadi masalah utama.** {n_sv:,} support vector x {n_features} dimensi:
  float32 = **{idn(sv_float32_mb)} MB**, float16 = **{idn(sv_float16_mb)} MB**, ditambah
  backbone ~3,7 MB. Totalnya sudah lewat batas 15 MB sebelum kode aplikasi dihitung.
- **Tidak ada operator SVM di TFLite.** Kernel RBF harus ditulis manual di Dart
  ({n_sv:,} dot product {n_features} dimensi untuk setiap foto). Jadi tidak bisa
  sekadar "salin `model.tflite` lalu jalan".
- Beban hitungnya sendiri masih wajar (~{flops/1e6:.0f} MFLOP, {lat_py*1000:.0f} ms di laptop
  ini), jadi yang memberatkan adalah ukuran dan kerumitan kode, bukan kecepatan.
- {kal_gain}

**Keputusan paket ini:** yang dikirim ke aplikasi adalah **CNN murni**
(`model.tflite` 4,6 MB, 1 inferensi, tanpa kode tambahan). SVM disimpan
sebagai jalur peningkatan v2 kalau nanti ponsel target sudah lebih kuat,
atau dipakai di sisi server.

Yang membuat CNN tetap layak: {kal_tutup}. {kal_banding}
"""
write_md(D / "DEPLOY.md", "Biaya Deploy Model 2 (SVM) di HP", [
    ("Ringkasan", body),
    ("Catatan pengukuran", f"""Latensi {idn(lat_py*1000)} ms/gambar diukur di laptop CPU ini,
bukan di HP. Angka itu hanya pembanding relatif, bukan janji performa.
Semua angka ukuran adalah perkiraan dari jumlah support vector x dimensi x
ukuran tipe data, tanpa kompresi."""),
])
print("DONE TAHAP 6b", flush=True)

  md -> C:\Users\Wahyu Pratomo\Project Machine Learning\tanaman-obat\output\02_svm\DEPLOY.md
DONE TAHAP 6b


---

# Selesai

Semua artefak ada di `output/`:

| Folder | Isi |
|---|---|
| `_data/` | manifest split |
| `01_cnn/` | model final, metrik, grafik, embedding |
| `02_svm/` | model 2 + `DEPLOY.md` |
| `03_rf/` | model 3 |
| `04_perbandingan/` | tabel + grafik perbandingan |
| `05_bahan_aplikasi/` | `model.tflite`, `labels.json`, `safety_rules.json`, README untuk pengembang |

Laporan lengkap: `output/RINGKASAN.md`.